### Create stac index
This notebook demonstrates how to create a STAC item for the Harmonized Landsat Sentinel-2 (HLS) Sentinel-2 product.

The implementation is tailored to a specific folder structure and may require adjustments to fit other use cases. The code assumes the following directory layout:

```
HLS.L30/  HLS.S30/
│
├── 17NRA/  19MFP/  ...
│   │
│   ├── 20210901-20220531/  20230901-20240531/  ...
│   │   │
│   │   ├── HLS.S30.T19MFP.2022001T144719.v2.0.B01.tif
│   │   └── HLS.S30.T19MFP.2022001T144719.v2.0.B02.tif
```

In [ ]:
# !pip install shapely
from osgeo import gdal
from collections import defaultdict
import json
import pystac
import os
import glob
import re
from datetime import datetime, timezone
from pystac.extensions.projection import ProjectionExtension
from pystac.extensions.raster import RasterExtension, RasterBand
from pystac.extensions.eo import EOExtension, Band
from shapely.geometry import shape

In [ ]:
class ProblemTracker:
    def __init__(self):
        self.errors = []

    def record(self, path, stage, message, exc=None, extra=None):
        self.errors.append({
            "path": os.path.abspath(path) if path else None,
            "stage": stage,
            "message": message,
            "exception": None if exc is None else str(exc),
            "extra": extra or {}
        })

    def save(self, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        with open(os.path.join(out_dir, "errors.json"), "w", encoding="utf-8") as f:
            json.dump(self.errors, f, indent=2)

    def __len__(self):
        return len(self.errors)

In [ ]:
class ResumeState:
    """Lightweight checkpoint for finished tiles."""
    def __init__(self, out_dir):
        self.path = os.path.join(out_dir, "_resume.json")
        self.state = {"completed_tiles": {}}
        try:
            with open(self.path, "r", encoding="utf-8") as f:
                self.state = json.load(f)
        except FileNotFoundError:
            pass

    def is_done(self, tile_id: str) -> bool:
        return tile_id in self.state.get("completed_tiles", {})

    def mark_done(self, tile_id: str, num_items: int):
        self.state.setdefault("completed_tiles", {})[tile_id] = {
            "finished_at": datetime.utcnow().isoformat() + "Z",
            "num_items": num_items
        }
        tmp = self.path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(self.state, f, indent=2)
        os.replace(tmp, self.path)

In [ ]:
def _load_existing_root(output_dir):
    cat_json = os.path.join(output_dir, "catalog.json")
    if os.path.exists(cat_json):
        try:
            return pystac.Catalog.from_file(cat_json)
        except Exception:
            pass
    return None

In [ ]:
def extract_scene_id(path_str):
    fname = os.path.basename(path_str)
    name = fname[:-4]                          # drop .tif
    return ".".join(name.split(".")[:-1])      # drop .Bxx

In [ ]:
def extract_tile_from_name(path_str):
    # HLS.L30.T33KVB.2023255T090122.v2.0.B06.tif -> T33KVB
    part = os.path.basename(path_str).split(".")[2]
    return part  # keep leading T to be unambiguous

In [ ]:
def find_timeslice_folder(rel_path):
    """
    Robust time slice finder when data_root is HLS.L30
    Searches parents for a folder that looks like YYYYMMDD-YYYYMMDD.
    Falls back to immediate parent if no match.
    """
    parts = rel_path.replace("\\", "/").split("/")
    # search from right to left, skipping the filename
    for comp in reversed(parts[:-1]):
        if re.fullmatch(r"\d{8}-\d{8}", comp):
            return comp
    # fallback: immediate parent
    parent = os.path.dirname(rel_path) or "root"
    return os.path.basename(parent) if parent != "root" else "root"

In [ ]:
def _read_gdal_info(file_path, problems: ProblemTracker):
    try:
        gdal.PushErrorHandler('CPLQuietErrorHandler')
        ds = gdal.Open(file_path)
        if ds is None:
            raise RuntimeError("GDAL could not open dataset")
        info_json = gdal.Info(ds, format='json')
        if not isinstance(info_json, dict):
            raise RuntimeError("gdal.Info returned non dict")
        # verify it is a GeoTIFF
        drv = info_json.get("driverShortName")
        if drv != "GTiff":
            raise RuntimeError(f"Not a GeoTIFF according to GDAL, driver={drv}")
        return ds, info_json
    except Exception as e:
        problems.record(file_path, "open", "Failed to read GeoTIFF with GDAL", e)
        return None, None

In [ ]:
# add near other globals
_TIME_RE = re.compile(
    r"""^
    (?P<ymdhms>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})
    (?:\.(?P<fraction>\d+))?
    (?P<tz>Z|[+\-]\d{2}:\d{2})?
    $""",
    re.X,
)

In [ ]:
def _parse_sensing_time(raw_value, file_path, problems: ProblemTracker):
    """Return timezone-aware datetime parsed from GDAL SENSING_TIME."""
    s = raw_value.split(";")[0].strip()
    # Normalize Z to offset so datetime.fromisoformat can parse it
    s = s[:-1] + "+00:00" if s.endswith("Z") else s
    m = _TIME_RE.match(s)
    if not m:
        problems.record(file_path, "time", "Unexpected SENSING_TIME format", extra={"value": raw_value})
        return None

    base = m.group("ymdhms")
    frac = (m.group("fraction") or "")[:6]
    tz = m.group("tz") or "+00:00"
    iso = f"{base}{('.' + frac) if frac else ''}{tz}"

    try:
        return datetime.fromisoformat(iso)
    except ValueError as e:
        problems.record(file_path, "time", "Could not parse SENSING_TIME", e, extra={"value": iso})
        return None

In [ ]:
def _parse_timeslice_range(s: str):
    m = re.fullmatch(r"(\d{8})-(\d{8})", s or "")
    if not m:
        return None, None
    start = datetime.strptime(m.group(1), "%Y%m%d").replace(tzinfo=timezone.utc)
    end   = datetime.strptime(m.group(2), "%Y%m%d").replace(tzinfo=timezone.utc)
    return start, end

In [ ]:
def _parse_datetime_from_name(filename: str):
    # Matches like .2023255T090122. (year=2023, day-of-year=255, time=090122)
    m = re.search(r"\.(\d{4})(\d{3})T(\d{6})\.", filename)
    if not m:
        return None
    year, doy, hms = m.group(1), m.group(2), m.group(3)
    try:
        return datetime.strptime(f"{year}{doy}{hms}", "%Y%j%H%M%S").replace(tzinfo=timezone.utc)
    except ValueError:
        return None

In [ ]:
def _parse_year_from_name(filename: str):
    m = re.search(r"\.(\d{4})(?:\.|$)", filename)
    if not m:
        return None
    y = int(m.group(1))
    return datetime(y, 1, 1, tzinfo=timezone.utc)

In [ ]:
def _epsg_from_info(info_json) -> int | None:
    """Return EPSG from GDAL Info JSON.
    Priority: (a) STAC proj:epsg, (b) metadata HORIZONTAL_CS_CODE='EPSG:xxxx'."""
    stac = info_json.get("stac", {})
    if stac.get("proj:epsg"):
        try:
            return int(stac["proj:epsg"])
        except (TypeError, ValueError):
            pass

    horiz = info_json.get("metadata", {}).get("", {}).get("HORIZONTAL_CS_CODE")
    if horiz:
        m = re.match(r"EPSG:(\d+)", horiz)
        if m:
            return int(m.group(1))
    return None

In [ ]:
def extract_metadata_gdal(file_path, problems: ProblemTracker):
    ds, info_json = _read_gdal_info(file_path, problems)
    if ds is None:
        return None, None, None, {}

    try:
        geometry = info_json["wgs84Extent"]
        bbox = list(shape(geometry).bounds)
    except Exception as e:
        problems.record(file_path, "extent", "Missing or invalid wgs84Extent", e, extra={"keys": list(info_json.keys())})
        return None, None, None, {}

    sensing_time = info_json.get("metadata", {}).get("", {}).get("SENSING_TIME")
    dt = _parse_sensing_time(sensing_time, file_path, problems) if sensing_time else None

    # start with whatever was in the embedded STAC block
    stac_fields = dict(info_json.get("stac", {}) or {})

    # ensure proj:epsg is present if we can infer it from metadata
    epsg = _epsg_from_info(info_json)
    if epsg is not None:
        stac_fields["proj:epsg"] = epsg  # keep as int

    return bbox, geometry, dt, stac_fields

In [ ]:
def get_raster_bands_info(file_path, problems: ProblemTracker):
    ds, _ = _read_gdal_info(file_path, problems)
    if ds is None:
        return None
    try:
        infos = []
        for i in range(1, ds.RasterCount + 1):
            b = ds.GetRasterBand(i)
            infos.append({
                "data_type": gdal.GetDataTypeName(b.DataType).lower(),
                "nodata": b.GetNoDataValue(),
                "scale": b.GetScale(),
                "offset": b.GetOffset()
            })
        return infos
    except Exception as e:
        problems.record(file_path, "band", "Failed to read raster band info", e)
        return None

In [ ]:
def create_asset(item, asset_key, file_path, stac_fields, problems: ProblemTracker):
    asset = pystac.Asset(
        href=os.path.abspath(file_path),
        media_type=pystac.MediaType.GEOTIFF,
        roles=["data"]
    )
    item.add_asset(asset_key, asset)

    try:
        eo = EOExtension.ext(item.assets[asset_key], add_if_missing=True)
        eo_bands = []
        for b in stac_fields.get("eo:bands", []):
            band = Band(properties={})
            if "name" in b:
                band.name = b["name"]
            if "description" in b:
                band.description = b["description"]
            eo_bands.append(band)
        eo.bands = eo_bands
    except Exception as e:
        problems.record(file_path, "eo", "Failed to attach EO bands", e)

    raster_band = get_raster_bands_info(file_path, problems)
    if raster_band is not None:
        item.assets[asset_key].extra_fields["raster:bands"] = raster_band

In [ ]:
def _first_good_metadata(tif_relpaths, data_root, problems: ProblemTracker, timeslice=None):
    for rel in tif_relpaths:
        path = os.path.join(data_root, rel)
        bbox, geometry, dt, stac_fields = extract_metadata_gdal(path, problems)
        if bbox and geometry:
            return bbox, geometry, dt, stac_fields, path
    return None, None, None, {}, None

In [ ]:
def create_stac_item(scene_id, tif_relpaths, data_root, problems: ProblemTracker, timeslice=None):
    # use the first file to extract shared metadata
    bbox, geometry, dt, stac_fields, first_used = _first_good_metadata(tif_relpaths, data_root, problems)
    if bbox is None:
        problems.record(scene_id, "item", "All bands in scene failed to yield base metadata", extra={"scene_files": tif_relpaths})
        return None

    item_id = os.path.basename(scene_id)
    
    # Try to keep exact dt if available
    exact_dt = dt
    if exact_dt is None:
        # try filename-based timestamp (e.g., L30 single-band names)
        try_rel = tif_relpaths[0] if tif_relpaths else None
        if try_rel:
            fname = os.path.basename(try_rel)
            exact_dt = _parse_datetime_from_name(fname)
            if exact_dt is None:
                exact_dt = _parse_year_from_name(fname)

    if exact_dt is not None:
        item = pystac.Item(
            id=item_id,
            geometry=geometry,
            bbox=bbox,
            datetime=exact_dt,
            properties={}
        )
    else:
        # fall back to a range from the folder like YYYYMMDD-YYYYMMDD
        start_dt, end_dt = _parse_timeslice_range(timeslice)
        if not (start_dt and end_dt):
            problems.record(scene_id, "time", "No datetime and no valid time range available",
                            extra={"timeslice": timeslice, "files": tif_relpaths})
            return None

        item = pystac.Item(
            id=item_id,
            geometry=geometry,
            bbox=bbox,
            datetime=None,
            start_datetime=start_dt,
            end_datetime=end_dt,
            properties={}
        )
    
    item.stac_extensions = [
        "https://stac-extensions.github.io/projection/v1.0.0/schema.json",
        "https://stac-extensions.github.io/eo/v1.1.0/schema.json",
        "https://stac-extensions.github.io/raster/v1.1.0/schema.json"
    ]

    # projection fields that are common to all bands
    try:
        proj = ProjectionExtension.ext(item, add_if_missing=True)
        epsg = stac_fields.get("proj:epsg")
        try:
            epsg = int(epsg) if epsg is not None else None
        except (TypeError, ValueError):
            epsg = None
        
        proj.epsg = epsg
        # this makes STAC Browser show a non-n/a "Code"
        if epsg:
            proj.code = f"EPSG:{epsg}"
    
        proj.shape = stac_fields.get("proj:shape")
        proj.transform = stac_fields.get("proj:transform")
    except Exception as e:
        problems.record(first_used or scene_id, "proj", "Failed to attach projection fields", e)

    # add one asset per band file
    good_assets = 0
    for rel in tif_relpaths:
        full = os.path.join(data_root, rel)
        # try to read eo fields first to confirm file health
        _, _, _, stac_fields_band = extract_metadata_gdal(full, problems)
        if stac_fields_band == {}:
            # already recorded inside extract_metadata_gdal
            continue
        band_id = os.path.splitext(os.path.basename(rel))[0].split(".")[-1].lower()
        try:
            create_asset(item, band_id, full, stac_fields_band, problems)
            good_assets += 1
        except Exception as e:
            problems.record(full, "asset", "Failed to attach asset to item", e)

    if good_assets == 0:
        problems.record(scene_id, "item", "No valid assets for this item")
        return None

    return item


In [ ]:
def create_catalog_from_folder(data_root, output_dir="tempo_stac", resume=True):
    """
    Creates a STAC catalog from an HLS.L30 folder tree
    Parameters:
        data_root (str): Path to the HLS.L30 root directory containing tile folders.
        output_dir (str): Directory where the generated STAC catalog will be saved.
    """
    problems = ProblemTracker()
    os.makedirs(output_dir, exist_ok=True)
    rs = ResumeState(output_dir) if resume else None

    # walk all tif under data_root, including all tiles like 33KVB, 33LUC, ...
    all_rel = [os.path.relpath(p, data_root)
               for p in glob.glob(os.path.join(data_root, "**", "*.tif"), recursive=True)]

    # tile -> timeslice -> scene -> [files]
    grouped = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for rel in all_rel:
        # guard name parsing
        try:
            tile = extract_tile_from_name(rel)        # e.g. T33KVB
            timeslice = find_timeslice_folder(rel)    # e.g. 20240701-20241130
            scene = extract_scene_id(rel)             # e.g. HLS.L30.T33KVB.2023...v2.0
            grouped[tile][timeslice][scene].append(rel)
        except Exception as e:
            problems.record(rel, "name-parse", "Failed to parse naming parts", e)

    # root catalog
    root = _load_existing_root(output_dir) or pystac.Catalog(id="stac-hls", description="HLS catalog grouped by tile and time slice")

    # tile level
    # e.g tile = T33KVB, by_time = 20230901-20230930
    for tile, by_time in grouped.items():
        # e.g. T33KVB -> 33KVB
        tile_label = tile[1:] if tile.startswith("T") else tile

        # resume skip: if already finished and found on disk, continue
        if resume and rs.is_done(tile_label):
            # ensure already present in loaded catalog; if not, rebuild this tile
            already = root.get_child(tile_label) if hasattr(root, "get_child") else None
            if already is not None:
                print(f"{tile_label} skip (already done)", flush=True)
                continue
            else:
                print(f"{tile_label} marked done but missing on disk, rebuilding", flush=True)

        tile_cat = root.get_child(tile_label) if hasattr(root, "get_child") else None
        if tile_cat is None:
            tile_cat = pystac.Catalog(id=tile_label, description=f"Tile {tile_label}")
            root.add_child(tile_cat)

        # time-slice level
        num_items_for_tile = 0
        # e.g. timeslice = 20230901-20230930, by_scene = HLS.L30.T33KVB.2023255T090122.v2.0
        for timeslice, by_scene in by_time.items():
            time_cat = tile_cat.get_child(timeslice) if hasattr(tile_cat, "get_child") else None
            if time_cat is None:
                time_cat = pystac.Catalog(id=timeslice, description=f"Time slice {timeslice}")
                tile_cat.add_child(time_cat)

            # items per scene
            # e.g. scene_id = HLS.S30.T36NUH.2022264T075619.v2.0
            # e.g. tif_relpaths = 36NUH/20240101-20240331/HLS.S30.T36NUH.2022264T075619.v2.0.B02.tif
            for scene_id, tif_relpaths in by_scene.items():
                item = create_stac_item(scene_id, tif_relpaths, data_root, problems, timeslice=timeslice)
                if item is not None:
                    time_cat.add_item(item)
                    num_items_for_tile += 1
                else:
                    problems.record(scene_id, "skip", "Skipped scene due to fatal errors")

        # progress print for this tile
        print(f"{tile_label} done", flush=True)

        # incremental save so work survives a crash and can be resumed
        root.normalize_and_save(output_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED, skip_unresolved=True)

        # mark checkpoint only after successful save
        if resume:
            rs.mark_done(tile_label, num_items_for_tile)

    # final write to be safe (cheap if nothing changed)
    root.normalize_and_save(output_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED, skip_unresolved=True)
    problems.save(output_dir)

    print(f"STAC written to: {os.path.abspath(output_dir)}")
    print(f"Error report: {os.path.abspath(os.path.join(output_dir, 'errors.json'))}")
    print(f"Total errors logged: {len(problems)}")

    return root

In [ ]:
# run
folder_path = "data/agb_temp/HLS.L30"
create_catalog_from_folder(data_root=folder_path, output_dir="data/stac_l30")

### How to Serve a STAC Catalog in a Simple Way

In [ ]:
import os
from http.server import HTTPServer, SimpleHTTPRequestHandler

os.chdir("data/stac_s30")


class CORSRequestHandler(SimpleHTTPRequestHandler):
    def end_headers(self) -> None:
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET")
        self.send_header("Cache-Control", "no-store, no-cache, must-revalidate")
        return super(CORSRequestHandler, self).end_headers()


with HTTPServer(("localhost", 5556), CORSRequestHandler) as httpd:
    httpd.serve_forever()